<a href="https://colab.research.google.com/github/rohith154-vsr/ML-2-LAB/blob/main/LP_3_FINDS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install ucimlrepo

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo

# load Car Evaluation dataset directly from UCI repository
car_evaluation = fetch_ucirepo(id=19)
X = car_evaluation.data.features.copy()
y = car_evaluation.data.targets.copy()

X.columns = ["buying", "maint", "doors", "persons", "lug_boot", "safety"]
y = y.iloc[:, 0]
df = X.copy()
df["Target"] = y

print("Dataset dimensions:", df.shape)
display(df.head())
print("
Target class distribution:
", df["Target"].value_counts())
print("
Null counts:
", df.isnull().sum())

# visualize class distribution
plt.figure(figsize=(7, 4))
sns.countplot(x="Target", data=df, hue="Target", palette="Set2", legend=False)
plt.title("Car Evaluation Target Distribution")
plt.xlabel("Evaluation Category")
plt.ylabel("Number of Cars")
plt.tight_layout()
plt.show()

features = ["buying", "maint", "doors", "persons", "lug_boot", "safety"]

# Find-S concept: focuses purely on the target positive class
# Negative examples are intentionally ignored because Find-S only generalizes from positive instances
positive_class = "vgood"
positive_data = df[df["Target"] == positive_class].copy()

print(f"Total positive examples ('{positive_class}'): {len(positive_data)}")
display(positive_data[features].head(10))

# initialize hypothesis to most specific: all values are null ('A~' / phi)
hypothesis = ["A~"] * len(features)
history = []

for _, row in positive_data.iterrows():
    example = [row[f] for f in features]
    
    # first positive example initializes the hypothesis
    if hypothesis == ["A~"] * len(features):
        hypothesis = example.copy()
    else:
        # for subsequent positive examples, replace any mismatch with general '?'
        for i in range(len(hypothesis)):
            if hypothesis[i] != example[i]:
                hypothesis[i] = "?"
                
    history.append(hypothesis.copy())

# record step-by-step hypothesis evolution
history_df = pd.DataFrame(history, columns=features)
history_df.index = range(1, len(history_df) + 1)

print("
Hypothesis Evolution across positive examples:")
print(history_df.to_string())

print("
Final Specific Hypothesis:")
for f, v in zip(features, hypothesis):
    print(f"  {f} = {v}")

# heatmap showing how specific feature constraints generalize over iterations
encoded = history_df.copy()
for col in encoded.columns:
    encoded[col] = encoded[col].astype("category").cat.codes

plt.figure(figsize=(10, 6))
sns.heatmap(encoded, annot=history_df.values, fmt='', cmap='YlGnBu',
            linewidths=0.4, linecolor='gray')
plt.title("Find-S Hypothesis Generalization Heatmap")
plt.xlabel("Features")
plt.ylabel("Positive Example Number")
plt.tight_layout()
plt.show()